# Item 09 — a matriz de avaliação e a ablação por tipo de aresta

Entrada: os 18 checkpoints de `item06_08_escada_treinos_colab.ipynb`.
Saída: as predições e os deltas de erro que o item 10 consolida.

Duas medições, nas mesmas condições para as três variantes:

1. **Modo 2, previsão genuína k passos à frente** (`k ∈ {1, 2, 4}`), com a janela ancorada na
   origem e nenhuma semana futura entrando no encoder. Cada predição é marcada como **on-chart**
   ou **piso**, porque ~95% dos alvos do span de teste estão no piso e a leitura on-chart é a
   principal ([ADR-0004](../docs/adr/0004-recorte-on-chart-como-leitura-principal.md)).
2. **Ablação por tipo de aresta e permutação por grupo de features** sobre a proposta, com o
   harness corrigido em `2c645c6`. O instrumento antigo encodava a semana alvo em vez da janela,
   `Δ` chegava constante ao GRU e **todo** delta saía exatamente zero
   ([`docs/diagnostico-ablacao.md`](../docs/diagnostico-ablacao.md)); por isso
   `results/phase3/interpretability.parquet` não pode ser citado.

**As duas leituras de orçamento de arestas.** O treino usa `max_cotraj_edges = 30_000` por
snapshot e a avaliação da qualificação usava o grafo completo (480k a 664k arestas). O modelo
nunca viu a topologia inteira, então um religamento aleatório pode simplesmente se parecer mais
com o que ele viu: é o confundidor apontado no ticket 08. Aqui cada modelo é avaliado nas duas,
grafo completo e mesmo teto do treino, e as duas são reportadas.

**Custo.** O Modo 2 é o passo caro: uma cópia isolada do banco de popularidade por origem, para
que origens da mesma música em semanas diferentes nunca contaminem o histórico "real" uma da
outra. A célula de calibração mede o tempo em 200 origens antes de você gastar a sessão inteira.
Tudo é retomável: cada combinação grava seu parquet e o laço pula os prontos.


## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.


In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")


In [ ]:
import json, time, shutil
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.interpretability import _empty_edge_store, predict_all
from music_diffusion_gnn.evaluation.longhits import onchart_weeks
from music_diffusion_gnn.evaluation.rollout import gnn_rollout_recursive
from music_diffusion_gnn.evaluation.stats import aggregate_seeds
from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.graph.controls import rewire_preserving_degree, strip_all_edges
from music_diffusion_gnn.models.diffusion_gnn import MusicDiffusionGNN
from music_diffusion_gnn.training.dataset import (
    _CHART_CODE, aggregate_weekly, build_pop_bank, build_samples, get_split_regime,
)

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IS_COLAB and DEVICE != "cuda":
    raise RuntimeError("GPU não ativada no Colab: Ambiente de execução → Alterar tipo de runtime → GPU.")
print("ROOT =", ROOT, "| DEVICE =", DEVICE)


In [ ]:
# Checkpoints vêm da pasta do notebook de treino; os resultados vão para uma pasta própria.
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/music-influence-gnn")
else:
    BASE = ROOT / "results"
CKPT_DIR = BASE / "escada_treinos"
OUT      = BASE / "avaliacao_matriz"
OUT.mkdir(parents=True, exist_ok=True)
print("checkpoints ←", CKPT_DIR, f"({len(list(CKPT_DIR.glob('gnn_*.pt')))} arquivos)")
print("artefatos   →", OUT)


## 1. Configuração

`ORIGIN_STRIDE = 4` é o passo usado na Phase 3, mantido para que as origens sejam do mesmo tipo
que as já reportadas. Se a calibração indicar que a matriz inteira não cabe no tempo disponível,
`8` corta as origens pela metade sem mudar nada mais.

A leitura de orçamento casado (`teto_treino`) roda só na primeira seed de cada variante e regime:
ela existe para responder se o descasamento treino/avaliação explica a prévia adversa do item 04,
não para entrar célula a célula na comparação final.


In [ ]:
MAX_WEEK   = 260
KS         = (1, 2, 4)
SEEDS      = [42, 43, 44]
REGIMES    = ["current", "pre_pandemia"]
VARIANTES  = ["proposta", "sem_grafo", "embaralhado"]

ORIGIN_STRIDE    = 4
ORCAMENTOS       = {"completo": None, "teto_treino": 30_000}
ORCAMENTO_TODAS_AS_SEEDS = "completo"     # o outro roda só na primeira seed
TEMPO_LIMITE_MIN = 150

# ablação (item 09) — roda sobre a proposta
SEEDS_ABLACAO      = [42, 43, 44]
TARGET_WEEK_STRIDE = 2        # semanas alvo do span de teste usadas na ablação
GRUPOS_FEATURES    = {"acustica": list(range(0, 9)), "metadados": [9, 10, 11]}
EPS                = 1e-9     # abaixo disto, "não mudou nada"

PLANO = [(v, r, s) for r in REGIMES for s in SEEDS for v in VARIANTES]
print(f"{len(PLANO)} modelos × {len(ORCAMENTOS)} leituras de orçamento (a segunda só na seed {SEEDS[0]})")


## 2. Dados, origens e o recorte on-chart

`onchart_weeks` é a mesma definição usada pelo `run_phase3.py`: a semana conta quando a música
esteve no chart em pelo menos um dia (`rank_score > 0`). O complemento é o piso, que é ausência de
observação e não popularidade baixa.

As origens são fixadas por regime e gravadas em disco: o item 10 refaz SIR e persistência **sobre
estas mesmas origens**, senão a comparação não é célula a célula.


In [ ]:
ts = pd.read_parquet(TS_PATH)
weekly = aggregate_weekly(ts)
onchart = onchart_weeks(ts, max_week=MAX_WEEK)
print(f"pares (música, chart, semana) on-chart: {len(onchart):,}")

def origens_regime(regime: str) -> pd.DataFrame:
    alvo = OUT / f"origens_{regime}.parquet"
    if alvo.exists():
        return pd.read_parquet(alvo)
    r = get_split_regime(regime)
    fim = r.test_end_week if r.test_end_week is not None else MAX_WEEK
    tw = weekly[(weekly["week"] >= r.test_start_week) & (weekly["week"] <= fim)
                & (weekly["week"] + max(KS) <= MAX_WEEK)]
    linhas = []
    for (sid, chart), grp in tw.groupby(["song_id", "chart"], observed=True):
        for w in sorted(grp["week"].unique())[::ORIGIN_STRIDE]:
            linhas.append({"song_id": sid, "chart": chart, "week": int(w)})
    o = pd.DataFrame(linhas, columns=["song_id", "chart", "week"])
    o.to_parquet(alvo, index=False)
    return o

for regime in REGIMES:
    o = origens_regime(regime)
    print(f"{regime}: {len(o):,} origens | {o.song_id.nunique():,} músicas | "
          f"semanas {o.week.min()}–{o.week.max()}")


## 3. Carregar um modelo e o grafo que ele viu

O modelo embaralhado precisa ser avaliado sobre o **mesmo** grafo embaralhado com que treinou, e o
sem-grafo sobre o grafo sem arestas. O checkpoint guarda a seed do religamento (`graph_seed`), que
é o que torna a topologia reproduzível a partir do arquivo.


In [ ]:
_cache_regime: dict[str, dict] = {}

def dados(regime: str) -> dict:
    if regime not in _cache_regime:
        g = torch.load(graph_path(regime, GRAPH_DIR), weights_only=False)
        _cache_regime[regime] = {
            "g": g,
            "pop_bank": build_pop_bank(weekly, NMAP_PATH, n_music=g["music"].num_nodes),
        }
    return _cache_regime[regime]

def grafo_da_variante(g, variante: str, graph_seed):
    if variante == "proposta":
        return g
    if variante == "sem_grafo":
        return strip_all_edges(g)
    if variante == "embaralhado":
        assert graph_seed is not None, "checkpoint embaralhado sem graph_seed: topologia irrecuperável"
        return rewire_preserving_degree(g, seed=int(graph_seed))
    raise ValueError(variante)

def carregar(variante: str, regime: str, seed: int):
    """Devolve (modelo pronto para eval, grafo da variante, metadados do checkpoint)."""
    ck_path = CKPT_DIR / f"gnn_{variante}_{regime}_seed{seed}.pt"
    if not ck_path.exists():
        return None, None, None
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    d = dados(regime)
    modelo = MusicDiffusionGNN(d["g"].metadata(), hidden=ck["hidden"], layers=ck["layers"],
                               dropout=ck["dropout"], pop_bank=d["pop_bank"])
    sd = dict(ck["state_dict"]); sd.pop("pop_bank", None)
    faltando, sobrando = modelo.load_state_dict(sd, strict=False)
    assert set(faltando) <= {"pop_bank"} and not sobrando, (faltando, sobrando)
    # A última camada da cabeça nasce zerada (Δ ≡ 0 → ŷ = persistência). Se o
    # checkpoint não carregou de verdade, ela continua zerada e TODA ablação sobre
    # ele devolve zero — que é justamente o achado que o item 04 provou ser
    # instrumento quebrado. Barrar aqui é mais barato que descobrir depois.
    assert modelo.head.mlp[-1].weight.abs().max().item() > 0, (
        f"{ck_path.name}: cabeça temporal zerada, o checkpoint não foi carregado")
    g_var = grafo_da_variante(d["g"], ck.get("variante", variante), ck.get("graph_seed"))
    return modelo.to(DEVICE).eval(), g_var, ck

_m, _g, _ck = carregar("proposta", "current", 42)
print("checkpoint de referência:", _ck["config_str"] if _ck else "AUSENTE — rode o notebook de treino primeiro")


## 4. Calibração — quanto custa o Modo 2 aqui

Roda 200 origens e extrapola. Se a projeção para a matriz inteira não couber nas sessões que você
tem, suba `ORIGIN_STRIDE` para 8 na célula de configuração e rode de novo: as origens são
regravadas e o item 10 usará as novas para todos os modelos, inclusive SIR e persistência.


In [ ]:
if _m is not None:
    amostra = origens_regime("current").head(200)
    t0 = time.time()
    _ = gnn_rollout_recursive(_m, _g, weekly, amostra, W=_ck["W"], ks=KS,
                              device=DEVICE, regime="current")
    por_origem = (time.time() - t0) / len(amostra)
    n_total = sum(len(origens_regime(r)) for r in REGIMES) * len(SEEDS) * len(VARIANTES)
    n_total += sum(len(origens_regime(r)) for r in REGIMES) * len(VARIANTES)  # leitura de orçamento
    print(f"{por_origem*1000:.1f} ms/origem  →  matriz completa ≈ {por_origem*n_total/3600:.1f} h de GPU")
    print(f"  (por modelo: {por_origem*len(origens_regime('current'))/60:.1f} min no regime current)")


## 5. Modo 2 — as predições de todos os modelos

Um parquet por combinação, com `y_true`, `y_pred` e a marca de recorte já resolvida. O laço pula o
que existe e para no limite de tempo: rode em quantas sessões precisar.


In [ ]:
def marcar_recorte(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["target_week"] = df["origin_week"] + df["k"]
    df["onchart"] = [
        (sid, chart, int(w)) in onchart
        for sid, chart, w in zip(df["song_id"], df["chart"], df["target_week"])
    ]
    return df

t_inicio = time.time()
for variante, regime, seed in PLANO:
    for nome_orc, orcamento in ORCAMENTOS.items():
        if nome_orc != ORCAMENTO_TODAS_AS_SEEDS and seed != SEEDS[0]:
            continue
        alvo = OUT / f"mode2_{variante}_{regime}_seed{seed}_{nome_orc}.parquet"
        if alvo.exists():
            print(f"{alvo.name}: pronto — pulando")
            continue
        if (time.time() - t_inicio) / 60 > TEMPO_LIMITE_MIN:
            print(f"\nlimite de {TEMPO_LIMITE_MIN} min atingido. Rode a célula de novo numa "
                  "sessão nova: o que já saiu será pulado.")
            break

        modelo, g_var, ck = carregar(variante, regime, seed)
        if modelo is None:
            print(f"{variante}/{regime}/seed{seed}: checkpoint ausente — pulando")
            continue
        t0 = time.time()
        df = gnn_rollout_recursive(modelo, g_var, weekly, origens_regime(regime),
                                   W=ck["W"], ks=KS, device=DEVICE, regime=regime,
                                   max_cotraj_edges=orcamento)
        df = marcar_recorte(df)
        df["variante"], df["split_regime"], df["seed"] = variante, regime, seed
        df["orcamento_arestas"] = nome_orc
        df["max_cotraj_edges"] = -1 if orcamento is None else orcamento
        df.to_parquet(alvo, index=False)
        print(f"{alvo.name}: {len(df):,} linhas, on-chart {df.onchart.mean():.1%} "
              f"[{(time.time()-t0)/60:.1f} min]")
        del modelo, g_var
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    else:
        continue
    break


## 6. Leitura rápida da matriz

RMSE por variante, horizonte, chart e recorte, com média e desvio entre seeds. Ainda não é a
comparação do item 10 (falta SIR, falta persistência, falta a estatística pareada), mas já mostra
a ordem entre as três variantes neurais, que é o que o pré-compromisso do
[ADR-0001](../docs/adr/0001-precompromisso-de-falseamento.md) lê.


In [ ]:
arquivos = sorted(OUT.glob("mode2_*.parquet"))
if not arquivos:
    raise SystemExit("nenhum parquet de Modo 2 ainda: rode a célula anterior")
m2 = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)
m2 = m2.dropna(subset=["y_true", "y_pred"])
print(f"{len(m2):,} predições | {m2.variante.nunique()} variantes | "
      f"{m2.groupby(['variante','split_regime']).seed.nunique().min()} seed(s) por célula (mínimo)")

CHAVE = ["variante", "split_regime", "orcamento_arestas", "chart", "k"]

def rmse_por_seed(df, recorte):
    """RMSE de cada (célula, seed). A agregação entre seeds vem depois: média de
    RMSE por seed, não RMSE de tudo empilhado, senão a dispersão some."""
    sub = df if recorte == "full" else df[df.onchart]
    e = sub.assign(err2=(sub.y_true - sub.y_pred) ** 2)
    out = e.groupby(CHAVE + ["seed"])["err2"].mean().pow(0.5).reset_index(name="rmse")
    out["recorte"] = recorte
    return out

por_seed = pd.concat([rmse_por_seed(m2, r) for r in ("full", "onchart")], ignore_index=True)
por_seed.to_parquet(OUT / "rmse_por_seed.parquet", index=False)

linhas = []
for chave, grp in por_seed.groupby(CHAVE + ["recorte"]):
    media, desvio = aggregate_seeds(grp["rmse"].to_numpy())
    linhas.append(dict(zip(CHAVE + ["recorte"], chave))
                  | {"rmse_medio": media, "rmse_desvio": desvio, "n_seeds": len(grp)})
resumo = pd.DataFrame(linhas)
resumo.to_parquet(OUT / "resumo_mode2.parquet", index=False)

principal = resumo[(resumo.orcamento_arestas == "completo") & (resumo.recorte == "onchart")]
display(principal.pivot_table(index=["split_regime", "chart", "k"], columns="variante",
                              values="rmse_medio").round(6))


In [ ]:
# As duas leituras de orçamento, pareadas na seed que rodou as duas (ticket 08)
comp = (por_seed[por_seed.seed == SEEDS[0]]
        .pivot_table(index=["variante", "split_regime", "recorte", "chart", "k"],
                     columns="orcamento_arestas", values="rmse")
        .dropna())
if not comp.empty and "teto_treino" in comp.columns:
    comp["dif"] = comp["teto_treino"] - comp["completo"]
    display(comp.round(6))
    print("\ndif > 0 significa que avaliar com o mesmo teto do treino PIORA o erro, isto é, "
          "o grafo\ncompleto não é o que desfavorece o modelo real. dif < 0 é o contrário, e é "
          "a leitura que\no ticket 08 precisa reportar junto da comparação principal.")
else:
    print("a leitura de orçamento casado ainda não rodou para nenhuma célula")


## 7. Ablação por tipo de aresta e por grupo de features (item 09)

Sobre a proposta, em tempo de avaliação, com o harness corrigido. Cada tipo de aresta é esvaziado
numa **cópia** do grafo e o erro é remedido, nos dois recortes e nos dois charts.

Duas ressalvas registradas junto com o número:

- um `delta_rmse` exatamente zero é reportado como **falha de sensibilidade**, não como ausência
  de efeito: foi exatamente essa a assinatura do instrumento quebrado do item 04;
- a permutação por grupo de features cobre as 12 colunas estáticas do nó de música (acústica e
  metadados). A popularidade defasada é injetada pelo banco de popularidade, que é também a
  âncora de persistência da cabeça temporal: permutá-la destruiria a âncora e mediria outra coisa.
  A decomposição `y_prev` contra `Δ` que responde por ela está no item 04.


In [ ]:
def amostras_teste(regime: str, W: int):
    r = get_split_regime(regime)
    fim = r.test_end_week if r.test_end_week is not None else MAX_WEEK
    tw = weekly[(weekly["week"] >= r.test_start_week) & (weekly["week"] <= fim)].copy()
    semanas = sorted(tw["week"].unique())[::TARGET_WEEK_STRIDE]
    tw = tw[tw["week"].isin(set(semanas))]
    fs = weekly.groupby(["song_id", "chart"], observed=True)["week"].min().to_dict()
    samples = build_samples(tw, W=W, node_id_map_path=NMAP_PATH, first_seen=fs)

    idx_para_song = {v: k for k, v in
                     json.load(open(NMAP_PATH))["music"]["spotify_id_to_idx"].items()}
    cod_para_chart = {v: k for k, v in _CHART_CODE.items()}
    meta = pd.DataFrame({
        "song_id": [idx_para_song[s.song_idx] for s in samples],
        "chart":   [cod_para_chart[s.chart] for s in samples],
        "target_week": [s.target_week for s in samples],
        "y_true":  [s.y for s in samples],
    })
    meta["onchart"] = [(r_.song_id, r_.chart, int(r_.target_week)) in onchart
                       for r_ in meta.itertuples()]
    return samples, meta

def rmse_np(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def deltas(meta, y_base, y_var, rotulo, analise):
    """delta_rmse do ŷ perturbado contra o ŷ de referência, por recorte × chart."""
    linhas = []
    for recorte in ("full", "onchart"):
        for chart in ("ambos", "viral50", "top200"):
            m = np.ones(len(meta), dtype=bool)
            if recorte == "onchart":
                m &= meta.onchart.values
            if chart != "ambos":
                m &= (meta.chart == chart).values
            if m.sum() == 0:
                continue
            y = meta.y_true.values[m]
            linhas.append({"analise": analise, "componente": rotulo, "recorte": recorte,
                           "chart": chart, "n": int(m.sum()),
                           "rmse_ref": rmse_np(y, y_base[m]),
                           "delta_rmse": rmse_np(y, y_var[m]) - rmse_np(y, y_base[m])})
    return linhas


In [ ]:
for regime in REGIMES:
    for seed in SEEDS_ABLACAO:
        alvo = OUT / f"ablacao_proposta_{regime}_seed{seed}.parquet"
        if alvo.exists():
            print(f"{alvo.name}: pronto — pulando")
            continue
        modelo, g_var, ck = carregar("proposta", regime, seed)
        if modelo is None:
            print(f"proposta/{regime}/seed{seed}: checkpoint ausente — pulando")
            continue
        samples, meta = amostras_teste(regime, ck["W"])
        print(f"\nproposta/{regime}/seed{seed}: {len(samples):,} amostras, "
              f"on-chart {meta.onchart.mean():.1%}")

        t0 = time.time()
        y_base = predict_all(modelo, g_var, samples)
        linhas = []
        for et in g_var.edge_types:
            y_abl = predict_all(modelo, _empty_edge_store(g_var, et), samples)
            linhas += deltas(meta, y_base, y_abl, f"sem_{et[1]}[{et[0]}->{et[2]}]",
                             "ablacao_tipo_aresta")
            print(f"  {str(et):<50s} [{time.time()-t0:5.0f}s]")

        rng = np.random.default_rng(seed)
        for nome, cols in GRUPOS_FEATURES.items():
            g_perm = g_var.clone()
            x = g_perm["music"].x.clone()
            perm = torch.from_numpy(rng.permutation(x.shape[0]))
            idx = torch.as_tensor(cols, dtype=torch.long)
            x[:, idx] = x[perm][:, idx]
            g_perm["music"].x = x
            y_perm = predict_all(modelo, g_perm, samples)
            linhas += deltas(meta, y_base, y_perm, nome, "permutacao_grupo_features")
            print(f"  permutação {nome:<38s} [{time.time()-t0:5.0f}s]")

        df = pd.DataFrame(linhas)
        df["split_regime"], df["seed"] = regime, seed
        df.to_parquet(alvo, index=False)
        del modelo, g_var
        if DEVICE == "cuda":
            torch.cuda.empty_cache()


In [ ]:
arquivos = sorted(OUT.glob("ablacao_proposta_*.parquet"))
if arquivos:
    abl = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)
    agg = []
    for chave, grp in abl.groupby(["analise", "componente", "split_regime", "recorte", "chart"]):
        media, desvio = aggregate_seeds(grp["delta_rmse"].to_numpy())
        agg.append(dict(zip(["analise", "componente", "split_regime", "recorte", "chart"], chave))
                   | {"delta_rmse_medio": media, "delta_rmse_desvio": desvio, "n_seeds": len(grp)})
    agg = pd.DataFrame(agg)
    agg.to_parquet(OUT / "ablacao_agregada.parquet", index=False)

    principal = agg[(agg.recorte == "onchart") & (agg.chart == "ambos")]
    display(principal.sort_values(["split_regime", "delta_rmse_medio"], ascending=[True, False])
                     .round(6))

    zerados = agg[agg.delta_rmse_medio.abs() < EPS]
    if len(zerados):
        print(f"\n⚠️ {len(zerados)} células com delta exatamente zero. Isto se reporta como FALHA "
              "DE SENSIBILIDADE\ndo instrumento, não como ausência de efeito (item 04):")
        display(zerados[["analise", "componente", "split_regime", "recorte", "chart"]])
    else:
        print("\nnenhum delta exatamente zero: o instrumento mede.")


## 8. O que levar de volta

Do Drive (`avaliacao_matriz/`):

- `origens_{regime}.parquet` — as origens do Modo 2, **obrigatórias** para o item 10 refazer SIR e
  persistência sobre as mesmas células;
- `mode2_{variante}_{regime}_seed{seed}_{orcamento}.parquet` — as predições, já marcadas por
  recorte;
- `resumo_mode2.parquet` — RMSE médio e desvio entre seeds por célula;
- `ablacao_proposta_{regime}_seed{seed}.parquet` e `ablacao_agregada.parquet` — o item 09.

Em seguida, `item10_comparacao_consolidada.ipynb`.
